# DNNs - Selective units
Identify and validate selective units in a diverse set of DNNs


In [ ]:
from facebody.config import DATA_ROOT
from facebody.model_sets import TRAINED, UNTRAINED, MODELS, DIET_MODELS, diet_dict
from facebody.dnn.floc import (
    DNNfloc, compute_sel_resp, compute_sel_dprime,
)
from facebody.plotting import (
    plot_perc_layers, stats_perc_layers,
    plot_perc_layers_by_model, plot_perc_layers_mixed,
    plot_dprime_dist, stats_dprime_dd,
    plot_resp_dist, plot_dprime_layers,
    plot_guided_gradcam, plot_guided_gradcam_diff,
    plot_perc_layers_by_diet,
)
from facebody.myutils.models import ModelLoader
from facebody.myutils.dataset import ImageDataLoader
from facebody.myutils.feature_extractor import FeatureExtractor

device = "cuda:0"

## Models
- Architectures: AlexNet, VGG16, ResNet50
- Training datasets: Ecoset, ImageNet
- Learning objectives: Supervised, Self-supervised, Untrained

In [ ]:
# Models
print(f"Trained (N={len(TRAINED)}): {list(TRAINED)}")
print(f"Untrained (N={len(UNTRAINED)}): {list(UNTRAINED)}")

## Identify selective units
- Functional localizer dataset containing images of faces, bodies, scenes, and objects (288 each), as well as scrambled images (144) (Stigliani et al., 2015)

### Selectivity for faces and bodies

In [ ]:
# Find selective units
for model, layers in MODELS.items():
    ml = ModelLoader(model, DATA_ROOT, device)

    fl = ImageDataLoader(
        root=DATA_ROOT / "datasets" / "floc",
        class_names=["face", "body", "scene", "object", "scrambled"],
        transform=ml.preprocess,
    )

    fe = FeatureExtractor(ml, layers)
    activs, labels = fe.extract(fl)

    dnn_floc = DNNfloc(
        model, layers,
        class_to_idx=fl.dataset.class_to_idx,
        single_cats=["face", "body"],
        mixed_tuples=[("face", "body")],
        solo_cats=["scene"],
    )
    dnn_floc.find_sel_units(activs, labels)

In [ ]:
# Fig. 1C: Plot percentage of selective units across layers
plot_perc_layers(TRAINED, out_path="main/perc_layers.pdf")
stats_perc_layers(TRAINED)

In [ ]:
# Fig. S1: Plot percentage of selective units across layers per model
plot_perc_layers_by_model(TRAINED, out_path="suppl/perc_layers_models.pdf")

In [ ]:
# Fig. S2A: Plot percentage of selective units across layers in untrained models
plot_perc_layers(UNTRAINED, out_path="suppl/perc_layers_untrained.pdf")

### Selectivity for other category combinations

In [ ]:
# Find mixed-selective units for other combinations
for model, layers in TRAINED.items():
    ml = ModelLoader(model, DATA_ROOT, device)

    fl = ImageDataLoader(
        root=DATA_ROOT / "datasets" / "floc",
        class_names=["face", "body", "scene", "object", "scrambled"],
        transform=ml.preprocess,
    )

    fe = FeatureExtractor(ml, layers)
    activs, labels = fe.extract(fl)

    dnn_floc = DNNfloc(
        model, layers,
        class_to_idx=fl.dataset.class_to_idx,
        single_cats=["face", "body", "scene"],
        mixed_tuples=None,
    )
    dnn_floc.find_sel_units(activs, labels, out_name="floc_res_mixed")

In [ ]:
# Fig. S2B: Plot percentage of mixed-selective units for other combinations across layers
plot_perc_layers_mixed(TRAINED, out_path="suppl/mixed_perc_layers.pdf")

## Validate selective units

### Validation dataset
- Validation dataset containing images of faces, bodies, and objects (150 each; Hemera Photo objects)

In [ ]:
# Confirm selective units
for model, layers in MODELS.items():
    ml = ModelLoader(model, DATA_ROOT, device)

    vl = ImageDataLoader(
        root=DATA_ROOT / "datasets" / "validation",
        transform=ml.preprocess,
    )

    fe = FeatureExtractor(ml, layers)
    activs, labels = fe.extract(vl)

    compute_sel_resp(
        model, activs, labels, class_to_idx=vl.dataset.class_to_idx,
        img_db="validation",
    )

    compute_sel_dprime(
        model, activs, labels, class_to_idx=vl.dataset.class_to_idx,
        img_db="validation", baseline_cat="object",
    )

In [ ]:
# Fig. 2A: Plot selectivity aggregated across layers
plot_dprime_dist(TRAINED, out_path="main/dprime_dist.pdf")
stats_dprime_dd(TRAINED)

In [ ]:
# Fig. S4: Plot mean unit response per category aggregated across layers
plot_resp_dist(TRAINED, out_path="suppl/resp_dist.pdf")

In [ ]:
# Fig. S5: Plot selectivity based on validation set across layers
# A: Trained models
plot_dprime_layers(TRAINED, out_path="suppl/dprime_layers_val.pdf")

# B: Untrained models
plot_dprime_layers(UNTRAINED, out_path="suppl/dprime_layers_val_untrained.pdf")

### Feature visualization
- Visualize feature preferences for each unit type using Guided Grad-CAM (Selvaraju et al., 2017)

In [ ]:
# Fig. 2B: Plot feature visualizations
layer = "conv5"
plot_guided_gradcam(
    model_name="alexnet_ecoset", target_layer=layer, device=device,
    img_path=DATA_ROOT / "datasets/guided_gradcam_examples_personal",
    out_path="main/guided_gradcam.pdf",
)

In [ ]:
# Fig. S6: Plot feature visualizations
layer = "conv5"
plot_guided_gradcam(
    model_name="alexnet_ecoset", target_layer=layer, device=device,
    img_path=DATA_ROOT / "datasets/guided_gradcam_examples", n_imgs=8,
    out_path="suppl/guided_gradcam.pdf",
)

In [ ]:
# Fig. S7: Plot difference maps of feature visualizations
layer = "conv5"
plot_guided_gradcam_diff(
    model_name="alexnet_ecoset", target_layer=layer, device=device,
    img_path=DATA_ROOT / "datasets/guided_gradcam_examples", n_imgs=8,
    out_path="suppl/guided_gradcam_diff.pdf",
)

### Selectivity for models with different training diets

In [ ]:
# Training diets, matched across AlexNet, VGG16, and ResNet50
diet_dicts = diet_dict()
print(list(diet_dicts))

In [ ]:
# Find selective units
for model, layers in DIET_MODELS.items():
    ml = ModelLoader(model, DATA_ROOT, device)

    fl = ImageDataLoader(
        root=DATA_ROOT / "datasets" / "floc",
        class_names=["face", "body", "scene", "object", "scrambled"],
        transform=ml.preprocess,
    )

    fe = FeatureExtractor(ml, layers)
    activs, labels = fe.extract(fl)

    dnn_floc = DNNfloc(
        model, layers,
        class_to_idx=fl.dataset.class_to_idx,
        single_cats=["face", "body"],
        mixed_tuples=[("face", "body")],
    )
    dnn_floc.find_sel_units(activs, labels)

In [ ]:
# Fig. A1: Plot change in percentage of selective units for different training diets
plot_perc_layers_by_diet(diet_dicts=diet_dicts, out_path="suppl/perc_layers_diets.pdf")